# Ridge regression: five key points

In [ ]:
import warnings
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes, make_regression
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
warnings.simplefilter("ignore")

## 1 and 2. Coefficients shrink, never to 0; the largest shrink most

In [ ]:
data = load_diabetes()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=2)
rows = {}
for alpha in [0, 0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000]:
    reg = Ridge(alpha=alpha).fit(X_train, y_train)
    rows[alpha] = list(reg.coef_.round(2)) + [round(reg.score(X_test, y_test), 3)]
pd.DataFrame(rows, index=data.feature_names + ["test R2"]).T

## 3. Bias and variance against alpha (bootstrap estimate)

In [ ]:
rng = np.random.default_rng(42)
X = 5 * rng.random((100, 1)) - 2
y = 0.7 * X[:, 0] ** 2 - 2 * X[:, 0] + 3 + rng.standard_normal(100)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=2)

def bias_variance(alpha, rounds=200):
    r = np.random.default_rng(123)
    preds = []
    for _ in range(rounds):
        i = r.integers(0, len(ytr), len(ytr))      # resample the training rows
        model = make_pipeline(PolynomialFeatures(degree=15), StandardScaler(), Ridge(alpha=alpha))
        preds.append(model.fit(Xtr[i], ytr[i]).predict(Xte))
    preds = np.array(preds)
    return ((preds - yte) ** 2).mean(), ((preds.mean(0) - yte) ** 2).mean(), preds.var(0).mean()

for alpha in [0.01, 0.1, 1, 10, 100, 1000]:
    loss, bias2, var = bias_variance(alpha)
    print(f"alpha {alpha:>7}: error {loss:7.2f}  bias2 {bias2:5.2f}  variance {var:6.2f}")

## 4. The lowest point of the loss moves towards 0

In [ ]:
X, y = make_regression(n_samples=100, n_features=1, n_informative=1, noise=20, random_state=13)
x = X.ravel()
for lam in [0, 10, 50, 100]:
    best_m = (x * (y + 2.29)).sum() / ((x * x).sum() + lam)    # intercept held at -2.29
    print(f"lambda {lam:3d}: lowest loss at m = {best_m:.2f}")